# Notebook 2 — Model Definition & Sanity Check
**What this notebook does:** Load `model.py` from Drive, instantiate the model, and confirm all shapes are correct before committing to a full training run.

**Runtime:** CPU is fine — no data is loaded, just shape checks.

**Outputs to Drive:**
```
ariel/
  model_config.json    ← saved here, loaded by NB3 and NB4
```

---
### What is frozen vs trained?
| Component | Status | Why |
|---|---|---|
| TinyLlama (LLM) | **Frozen** — in memory, weights never change | Forward pass only for text encoding |
| PerceiverResampler | **Trained** — gradients flow here | Learns to compress frame embeddings |
| GatedCrossAttentionLayer | **Trained** | Learns to inject visual info into text |
| video_proj / text_proj | **Trained** | Learns shared retrieval embedding space |
| logit_scale | **Trained** | Learns contrastive temperature |
---

In [1]:
!pip install -q transformers accelerate torch

## Step 1 · Mount Drive and load model.py

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
PROJECT = '/content/drive/Shareddrives/DATA 298A/ariel'

# Add project folder to path so we can import model.py directly
sys.path.insert(0, PROJECT)
from model import build_model, DEFAULT_CONFIG

print('model.py loaded successfully.')
print('Default config:', DEFAULT_CONFIG)

Mounted at /content/drive
model.py loaded successfully.
Default config: {'llm_name': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0', 'perceiver_latents': 64, 'perceiver_depth': 6, 'xattn_heads': 8, 'xattn_every_n': 2, 'clip_dim': 768, 'proj_dim': 512}


## Step 2 · Save model_config.json to Drive

This is the single config file that NB3 and NB4 will load. Edit values here if you want to experiment with different sizes.

In [3]:
import json

# Edit these if you want to change the architecture
MODEL_CONFIG = {
    'llm_name'          : 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    'perceiver_latents' : 64,
    'perceiver_depth'   : 6,
    'xattn_heads'       : 8,
    'xattn_every_n'     : 2,
    'clip_dim'          : 768,
    'proj_dim'          : 512,
}

cfg_path = f'{PROJECT}/model_config.json'
with open(cfg_path, 'w') as f:
    json.dump(MODEL_CONFIG, f, indent=2)

print(f'model_config.json saved to {cfg_path}')
print('NB3 and NB4 will load this file automatically.')

model_config.json saved to /content/drive/Shareddrives/DATA 298A/ariel/model_config.json
NB3 and NB4 will load this file automatically.


## Step 3 · Build model and inspect parameter counts

In [4]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

model, tokenizer = build_model(MODEL_CONFIG, device)

total      = sum(p.numel() for p in model.parameters())
trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen     = total - trainable

print(f'\nTotal parameters : {total:>12,}')
print(f'Frozen (LLM)     : {frozen:>12,}  ← in GPU memory, never updated')
print(f'Trainable        : {trainable:>12,}  ← Perceiver + xattn + heads')

Device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


Total parameters : 1,715,360,279
Frozen (LLM)     : 1,100,048,384  ← in GPU memory, never updated
Trainable        :  615,311,895  ← Perceiver + xattn + heads


## Step 4 · End-to-end shape test with fake data

In [5]:
# Simulate a batch of 2 videos, 8 frames each, 768-dim CLIP embeddings
dummy_vis   = torch.randn(2, 8, 768).to(device)
dummy_masks = torch.ones(2, 8).to(device)
dummy_toks  = tokenizer(
    ['a cat sitting on a mat', 'person running in the park'],
    return_tensors='pt', padding=True, truncation=True, max_length=32
).to(device)

loss, vid_emb, txt_emb = model(
    dummy_vis, dummy_masks,
    dummy_toks['input_ids'], dummy_toks['attention_mask']
)

print(f'Loss        : {loss.item():.4f}  (random weights — value does not matter)')
print(f'Video embed : {vid_emb.shape}')   # expect (2, 512)
print(f'Text embed  : {txt_emb.shape}')   # expect (2, 512)
print('\nAll shapes correct — ready for NB3 training.')

Loss        : 0.6927  (random weights — value does not matter)
Video embed : torch.Size([2, 512])
Text embed  : torch.Size([2, 512])

All shapes correct — ready for NB3 training.
